# 💼 Freelancer Income & Experience — Statistics Notebook
**Dataset:** 1,000 simulated freelancers across tech disciplines

**Variables:** Experience, Hourly Rate, Weekly Hours, Client Rating, Country, Skill, Remote Status, Monthly Income

Full statistical workflow:
EDA → Descriptive Stats → Histograms → Cross-tabulation & Pivot Tables → Boxplots & Outliers → Count Plots → Central Tendency & Dispersion → Skewness → Z-Scores & p-Values → Chebyshev's Rule → t-Tests → Confidence Intervals → Correlation → Scatter & Pair Plots → Distribution Plots → Linear Regression

## 1. Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
%matplotlib inline

# Generate synthetic freelancer dataset (1000 records)
np.random.seed(42)
n = 1000

countries = ['USA', 'UK', 'Germany', 'Canada', 'Australia', 'Netherlands', 'UAE', 'India']
skills = ['Motion Design', 'Web Development', 'Data Science', 'UX/UI Design', 'Copywriting', 'Video Editing', 'DevOps', 'AI/ML']

experience_years = np.random.randint(0, 15, n)
hourly_rate = np.round(20 + experience_years * 5 + np.random.normal(0, 12, n), 2).clip(15, 200)
weekly_hours = np.random.randint(10, 60, n)
client_rating = np.round(np.random.uniform(3.0, 5.0, n), 2)
monthly_income = np.round(hourly_rate * weekly_hours * 4.33, 2)
remote = np.random.choice(['Yes', 'No'], n, p=[0.78, 0.22])
country = np.random.choice(countries, n)
skill = np.random.choice(skills, n)

mydata = pd.DataFrame({
    'Freelancer_ID': range(1, n+1),
    'Skill': skill,
    'Country': country,
    'Experience_Years': experience_years,
    'Hourly_Rate_USD': hourly_rate,
    'Weekly_Hours': weekly_hours,
    'Client_Rating': client_rating,
    'Remote': remote,
    'Monthly_Income_USD': monthly_income
})

print('Dataset created:', mydata.shape)

In [ ]:
mydata.head(10)

In [ ]:
mydata.tail(5)

In [ ]:
mydata.info()

In [ ]:
mydata.describe().round(2)

In [ ]:
print('=== Skill Distribution ===')
print(mydata['Skill'].value_counts())
print()
print('=== Country Distribution ===')
print(mydata['Country'].value_counts())
print()
print('=== Remote Work ===')
print(mydata['Remote'].value_counts())
print()
print('Missing Values:')
print(mydata.isnull().sum())

## 2. Histograms

In [ ]:
mydata[['Hourly_Rate_USD', 'Monthly_Income_USD']].hist(figsize=(12, 5))
plt.suptitle('Histograms of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Skill', column='Hourly_Rate_USD', figsize=(16, 10))
plt.suptitle('Hourly Rate Distribution by Skill', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Remote', column='Hourly_Rate_USD', figsize=(10, 5))
plt.suptitle('Hourly Rate Distribution by Remote Status', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Skill', column='Experience_Years', figsize=(16, 10))
plt.suptitle('Experience Distribution by Skill', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Cross-tabulation & Pivot Tables

In [ ]:
mydata['income_group'] = mydata['Monthly_Income_USD'].apply(lambda x: 'high' if x >= 5000 else 'low')
pd.crosstab(mydata['Skill'], mydata['income_group'], margins=True)

In [ ]:
mydata['exp_group'] = mydata['Experience_Years'].apply(lambda x: 'senior' if x >= 10 else ('mid' if x >= 5 else 'junior'))
mydata['rate_group'] = mydata['Hourly_Rate_USD'].apply(lambda x: 'high' if x >= 65 else 'low')
pd.pivot_table(mydata, index=['Skill', 'exp_group'], columns=['rate_group'], aggfunc=len, fill_value=0)

In [ ]:
pd.pivot_table(mydata, values='Hourly_Rate_USD',
               index=['Skill'],
               columns=['Remote'],
               aggfunc='mean').round(2)

## 4. Box Plots & Outlier Detection

In [ ]:
sns.boxplot(x='Remote', y='Hourly_Rate_USD', data=mydata)
plt.title('Hourly Rate by Remote Status')
plt.show()

In [ ]:
order = mydata.groupby('Skill')['Hourly_Rate_USD'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=mydata, x='Skill', y='Hourly_Rate_USD', order=order, palette='Blues_d', ax=ax)

ax.set_title('Hourly Rate Distribution by Skill', fontsize=13, fontweight='bold')
ax.set_xlabel('Skill')
ax.set_ylabel('Hourly Rate (USD)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\nMedian Hourly Rate by Skill:')
print(mydata.groupby('Skill')['Hourly_Rate_USD'].median().sort_values(ascending=False).round(2))

In [ ]:
# IQR Outlier Detection on Monthly Income
q1, q3 = np.percentile(mydata['Monthly_Income_USD'], [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = mydata[(mydata['Monthly_Income_USD'] < lower) | (mydata['Monthly_Income_USD'] > upper)]
print(f'Q1: ${q1:,.2f}, Q3: ${q3:,.2f}, IQR: ${iqr:,.2f}')
print(f'Lower bound: ${lower:,.2f}, Upper bound: ${upper:,.2f}')
print(f'Number of outliers: {len(outliers)}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(mydata['Monthly_Income_USD'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightsteelblue', color='black'),
            flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
plt.title('Box Plot: Monthly Income (USD)')
plt.xlabel('Monthly Income (USD)')
plt.show()

## 5. Count Plots

In [ ]:
sns.countplot(x='Skill', hue='income_group', data=mydata)
plt.title('High vs Low Income by Skill')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(x='Country', hue='rate_group', data=mydata)
plt.title('High vs Low Hourly Rate by Country')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Measures of Central Tendency & Dispersion

In [ ]:
income = mydata['Monthly_Income_USD']
hourly = mydata['Hourly_Rate_USD']

mean_income = np.mean(income)
std_income = np.std(income)
mean_rate = np.mean(hourly)
std_rate = np.std(hourly)

print(f'=== Monthly Income (USD) ===')
print(f'Mean:     ${mean_income:,.2f}')
print(f'Std Dev:  ${std_income:,.2f}')
print()
print(f'=== Hourly Rate (USD) ===')
print(f'Mean:     ${mean_rate:,.2f}')
print(f'Std Dev:  ${std_rate:,.2f}')

In [ ]:
q1, q2, q3 = np.percentile(mydata['Monthly_Income_USD'], [25, 50, 75])
print(f'Monthly Income Percentiles:')
print(f'Q1: ${q1:,.2f}  Median (Q2): ${q2:,.2f}  Q3: ${q3:,.2f}')

In [ ]:
print('=== Mean of Numeric Columns ===')
print(mydata.mean(numeric_only=True).round(2))
print()
print('=== Std Dev of Numeric Columns ===')
print(mydata.std(numeric_only=True).round(2))

## 7. Skewness & Distribution Shapes

In [ ]:
normal_dist = np.random.normal(mean_income, std_income, 500)
skewed_dist = st.skewnorm.rvs(a=10, loc=mean_income, scale=std_income, size=500)

plt.figure(figsize=(10, 5))
plt.hist(normal_dist, bins=30, alpha=0.5, label='Normal', color='skyblue')
plt.hist(skewed_dist, bins=30, alpha=0.5, label='Right Skewed', color='salmon')
plt.title('Normal vs Right-Skewed Distribution — Monthly Income')
plt.xlabel('Monthly Income (USD)')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plt.axvline(x=mydata['Monthly_Income_USD'].mean(), color='orange', label='Mean')
plt.axvline(x=np.median(mydata['Monthly_Income_USD']), color='green', label='Median')
plt.hist(mydata['Monthly_Income_USD'], color='lightgray', bins=30)
plt.title('Monthly Income: Mean vs Median')
plt.xlabel('Monthly Income (USD)')
plt.legend()
plt.show()

print(f'Skewness: {income.skew():.4f}')
print(f'Kurtosis: {income.kurtosis():.4f}')

## 8. Normal Distribution & Z-Score Visualization
Where does a **$95/hr** freelancer fall relative to the market?

**Formula:** Z = (X - μ) / σ

In [ ]:
x_i = 95    # target freelancer hourly rate
mu = 65     # population mean
sigma = 18  # population std dev

x = np.random.normal(mu, sigma, 10000)

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(x, color='lightsteelblue', stat='density', ax=ax)

ax.axvline(mu, color='orange', linewidth=2, label=f'Mean (μ = ${mu})')

for v in [-3, -2, -1, 1, 2, 3]:
    ax.axvline(mu + v * sigma, color='olivedrab', linewidth=1, linestyle='--')
    ax.text(mu + v * sigma, ax.get_ylim()[1] * 0.02, f'{v}σ',
            ha='center', fontsize=8, color='olivedrab')

ax.axvline(x_i, color='purple', linewidth=2.5, label=f'Target (${x_i}/hr)')

z = (x_i - mu) / sigma
p = st.norm.cdf(z)

ax.set_title(
    f'Z-Score Visualization — Hourly Rate\n'
    f'Z = ({x_i} - {mu}) / {sigma} = {z:.2f}  |  Percentile: {p*100:.1f}%',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Hourly Rate (USD)')
ax.set_ylabel('Density')
ax.legend()
ax.set_xlim(0, 140)

print(f'Z-Score for ${x_i}/hr:  {z:.4f}')
print(f'Percentile:            {p*100:.2f}%')
print(f'Interpretation:        This freelancer earns more than {p*100:.1f}% of the market.')

plt.tight_layout()
plt.show()

In [ ]:
z_scores = st.zscore(mydata['Hourly_Rate_USD'])
print('First 10 z-scores (Hourly Rate):', z_scores[:10].round(4))

In [ ]:
z_input = 1.5
x_val = mean_rate + z_input * std_rate
print(f'Raw hourly rate for Z={z_input}: ${x_val:.2f}')

prob = st.norm.cdf(50, loc=mean_rate, scale=std_rate)
print(f'P(Hourly Rate < $50): {prob:.4f}')

p_below = st.norm.cdf(-2.5)
p_above = 1 - st.norm.cdf(2.5)
print(f'P-value (|Z| > 2.5): {p_below + p_above:.4f}')

## 9. Chebyshev's Rule

In [ ]:
def chebyshev_rule(k):
    if k <= 1:
        return 'k must be greater than 1'
    return (1 - (1 / k**2)) * 100

print(f'At least {chebyshev_rule(2):.2f}% within 2 SD')
print(f'At least {chebyshev_rule(3):.2f}% within 3 SD')

within_k2 = mydata[
    (mydata['Monthly_Income_USD'] >= mean_income - 2*std_income) &
    (mydata['Monthly_Income_USD'] <= mean_income + 2*std_income)
]
print(f'Actual within 2 SD (Monthly Income): {100*len(within_k2)/len(mydata):.2f}%')

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(mydata['Monthly_Income_USD'], bins=30, color='gray', alpha=0.4)
plt.axvline(mean_income, color='blue', linestyle='dashed', label='Mean')
plt.axvline(mean_income - 2*std_income, color='green', linestyle='--', label='k=2 Lower')
plt.axvline(mean_income + 2*std_income, color='green', linestyle='--', label='k=2 Upper')
plt.title("Chebyshev's Theorem – Monthly Income")
plt.xlabel('Monthly Income (USD)')
plt.legend()
plt.show()

## 10. t-Tests

In [ ]:
# One-sample t-test: Is the mean hourly rate different from $60?
null_mean = 60.0
t_stat, p_val = st.ttest_1samp(mydata['Hourly_Rate_USD'], null_mean)
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value    : {p_val:.4f}')
alpha = 0.05
if p_val < alpha:
    print(f'Decision: Reject H0 – mean hourly rate differs significantly from ${null_mean}')
else:
    print('Decision: Fail to reject H0')

In [ ]:
# Two-sample t-test: Remote vs Non-Remote hourly rates
remote_rates = mydata[mydata['Remote'] == 'Yes']['Hourly_Rate_USD'].values
onsite_rates = mydata[mydata['Remote'] == 'No']['Hourly_Rate_USD'].values
t2, p2 = st.ttest_ind(remote_rates, onsite_rates, equal_var=False)
print(f'Remote mean hourly rate  : ${remote_rates.mean():.2f}')
print(f'On-site mean hourly rate : ${onsite_rates.mean():.2f}')
print(f'T-statistic: {t2:.4f},  P-value: {p2:.4f}')

## 11. Confidence Intervals

In [ ]:
n_samp = mydata['Monthly_Income_USD'].size
s = mydata['Monthly_Income_USD'].std()
xbar = mydata['Monthly_Income_USD'].mean()
z = 1.96

CI_err = z * (s / n_samp**0.5)
print(f'95% CI for Mean Monthly Income: (${xbar - CI_err:,.2f}, ${xbar + CI_err:,.2f})')

fig, ax = plt.subplots()
plt.ylabel('Monthly Income (USD)')
plt.grid(axis='y')
ax.errorbar(['Monthly Income'], [xbar], [CI_err], fmt='o', color='steelblue')
plt.title('95% Confidence Interval for Mean Monthly Income')
plt.show()

## 12. Correlation Analysis

In [ ]:
corr = mydata.corr(numeric_only=True)
corr.round(4)

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap — Freelancer Data')
plt.show()

In [ ]:
r, p = st.pearsonr(mydata['Experience_Years'], mydata['Hourly_Rate_USD'])
print(f'Pearson r (Experience vs Hourly Rate) = {r:.4f},  p-value = {p:.4e}')
print(f'R-squared = {r**2:.4f}')

## 13. Scatter Plots & Pair Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_exp = mydata['Experience_Years'].corr(mydata['Hourly_Rate_USD'])

fig, ax = plt.subplots(figsize=(9, 5))
sns.regplot(
    data=mydata, x='Experience_Years', y='Hourly_Rate_USD',
    scatter_kws={'alpha': 0.3, 'color': 'steelblue'},
    line_kws={'color': 'orangered', 'linewidth': 2},
    ax=ax
)
ax.set_title(f'Experience vs Hourly Rate  |  r = {corr_exp:.2f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Hourly Rate (USD)')
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(x='Experience_Years', y='Monthly_Income_USD', hue='Remote', data=mydata, alpha=0.5)
plt.title('Experience vs Monthly Income by Remote Status')
plt.show()

In [ ]:
sns.pairplot(mydata[['Experience_Years', 'Hourly_Rate_USD', 'Monthly_Income_USD', 'Remote']], hue='Remote')
plt.show()

## 14. Distribution Plots

In [ ]:
sns.displot(mydata['Hourly_Rate_USD'], kde=True, color='steelblue')
plt.title('Distribution of Hourly Rate (USD)')
plt.show()

In [ ]:
sns.displot(mydata['Monthly_Income_USD'], kde=True, color='darkorange')
plt.title('Distribution of Monthly Income (USD)')
plt.show()

In [ ]:
sns.displot(mydata['Experience_Years'], kde=True, color='seagreen')
plt.title('Distribution of Experience (Years)')
plt.show()

## 15. Linear Regression (OLS)
Does experience predict hourly rate? Manual OLS + statsmodels summary.

In [ ]:
x = mydata['Experience_Years'].values
y = mydata['Hourly_Rate_USD'].values

cov_mat = np.cov(x, y)
beta1 = cov_mat[0, 1] / cov_mat[0, 0]
beta0 = y.mean() - beta1 * x.mean()
print(f'Intercept (β0): {beta0:.4f}')
print(f'Slope     (β1): {beta1:.4f}')
print(f'Interpretation: Each additional year of experience adds ${beta1:.2f}/hr on average.')

In [ ]:
xline = np.linspace(x.min(), x.max(), 1000)
yline = beta0 + beta1 * xline

sns.scatterplot(x=x, y=y, alpha=0.4, color='steelblue')
plt.plot(xline, yline, color='orangered', linewidth=2)
plt.xlabel('Experience (Years)')
plt.ylabel('Hourly Rate (USD)')
plt.title('Linear Regression: Experience → Hourly Rate')
plt.show()

In [ ]:
import statsmodels.api as sm

X_ols = sm.add_constant(mydata[['Experience_Years']])
model = sm.OLS(mydata['Hourly_Rate_USD'], X_ols)
result = model.fit()
print(result.summary())